## Setup and Dependencies

In [1]:
import io
import os
import time
from pathlib import Path
from typing import Any

import fitz  # PyMuPDF  # type: ignore[import-untyped]
from azure.ai.documentintelligence import DocumentIntelligenceClient
from azure.core.credentials import AzureKeyCredential
from dotenv import load_dotenv
from PIL import Image

In [2]:
# Load environment variables
load_dotenv()

# Azure Document Intelligence configuration
AZURE_DI_ENDPOINT = os.getenv("AZURE_DI_ENDPOINT")
AZURE_DI_KEY = os.getenv("AZURE_DI_KEY")
AZURE_DI_MODEL_ID = os.getenv("AZURE_DI_MODEL_ID", "prebuilt-layout")

# Configuration
PDF_RENDER_DPI = 200  # DPI for PDF rendering
PADDING_PIXELS = 4  # Extra padding around crop

# Serverless compatibility: Only save to disk if explicitly enabled
SAVE_CROPS = os.getenv("SAVE_CROPS", "false").lower() == "true"

if not AZURE_DI_ENDPOINT or not AZURE_DI_KEY:
    print("⚠️ Warning: Azure Document Intelligence credentials not configured!")
    print("Please set AZURE_DI_ENDPOINT and AZURE_DI_KEY in your .env file")
else:
    print("✓ Azure Document Intelligence configured")
    print(f"  Endpoint: {AZURE_DI_ENDPOINT}")
    print(f"  Model: {AZURE_DI_MODEL_ID}")
    print(f"  Save crops to tmp/: {'Enabled' if SAVE_CROPS else 'Disabled (serverless mode)'}")

✓ Azure Document Intelligence configured
  Endpoint: https://r0baid0c.cognitiveservices.azure.com/
  Model: valid_id
  Save crops to tmp/: Enabled


## Helper Functions

In [3]:
def convert_pdf_to_image_bytes(file_path: str, dpi: int = 200) -> bytes:
    """
    Convert PDF first page to PNG image bytes, or return original bytes if already an image.
    
    Args:
        file_path: Path to PDF or image file
        dpi: DPI for PDF rendering (default: 200)
    
    Returns:
        PNG image bytes
    """
    file_ext = Path(file_path).suffix.lower()
    
    if file_ext == ".pdf":
        # Read PDF file
        with open(file_path, "rb") as f:
            file_bytes = f.read()
        
        # Extract first page of PDF as PNG image bytes
        pdf_document: Any = fitz.open(stream=file_bytes, filetype="pdf")
        if pdf_document.page_count == 0:
            raise ValueError("PDF has no pages")
        
        # Render first page to image at specified DPI
        page: Any = pdf_document[0]
        pix: Any = page.get_pixmap(dpi=dpi)
        png_bytes: bytes = pix.tobytes("png")
        return png_bytes
    else:
        # Already an image, read and return as-is
        with open(file_path, "rb") as f:
            return f.read()


def polygon_to_bbox(points: list[float]) -> tuple[float, float, float, float]:
    """Convert polygon [x1,y1,x2,y2,...] -> (min_x, min_y, max_x, max_y)."""
    xs = points[::2]
    ys = points[1::2]
    return min(xs), min(ys), max(xs), max(ys)


def crop_image_region(
    image_bytes: bytes, bbox: tuple[float, float, float, float], padding: int = 0
) -> bytes:
    """
    Crop region from image bytes.
    
    Args:
        image_bytes: Image file bytes
        bbox: Bounding box (min_x, min_y, max_x, max_y)
        padding: Padding in pixels
    
    Returns:
        PNG bytes
    """
    # Load image from bytes
    im = Image.open(io.BytesIO(image_bytes)).convert("RGBA")
    min_x, min_y, max_x, max_y = bbox
    
    box = (
        max(int(min_x) - padding, 0),
        max(int(min_y) - padding, 0),
        min(int(max_x) + padding, im.width),
        min(int(max_y) + padding, im.height),
    )
    crop = im.crop(box)
    
    # Convert to PNG bytes
    img_byte_arr = io.BytesIO()
    crop.save(img_byte_arr, format='PNG')
    png_bytes = img_byte_arr.getvalue()
    
    return png_bytes

## Azure Document Intelligence Signature Extraction

In [13]:
async def extract_signatures_with_doc_intelligence(
    file_path: str,
    model_id: str = AZURE_DI_MODEL_ID,
    output_dir: str = "../tmp"
) -> dict[str, Any]:
    """
    Extract signature regions using Azure Document Intelligence.
    
    SERVERLESS COMPATIBLE: All processing is done in-memory.
    File saving only happens if SAVE_CROPS=true environment variable is set.
    
    Args:
        file_path: Path to image or PDF file
        model_id: Document Intelligence model ID (default: prebuilt-layout)
        output_dir: Directory to save cropped images (only used if SAVE_CROPS=true)
    
    Returns:
        Dictionary containing extraction results (image_bytes in memory)
    """
    if not AZURE_DI_ENDPOINT or not AZURE_DI_KEY:
        raise ValueError("Azure Document Intelligence not configured")
    
    # Initialize client
    client = DocumentIntelligenceClient(
        AZURE_DI_ENDPOINT, AzureKeyCredential(AZURE_DI_KEY)
    )
    
    # Convert file to image bytes (PDFs are converted to images)
    print(f"Processing file: {file_path}")
    image_bytes = convert_pdf_to_image_bytes(file_path, dpi=PDF_RENDER_DPI)
    
    # Analyze document (SDK call is synchronous, so we don't await it)
    print(f"Analyzing with model: {model_id}")
    image_stream = io.BytesIO(image_bytes)
    poller = client.begin_analyze_document(
        model_id,
        body=image_stream,
        content_type="application/octet-stream"
    )
    result: Any = poller.result()
    
    # Collect signature regions
    regions: list[tuple[str, int, list[float]]] = []
    
    # Strategy 1: Try to find Signature fields from custom model
    documents_attr: Any = getattr(result, "documents", None)
    if documents_attr:
        for doc in documents_attr:
            fields_attr: Any = getattr(doc, "fields", None)
            if fields_attr:
                for field_name, field_value in fields_attr.items():
                    fname_str: str = str(field_name)
                    if "signature" in fname_str.lower():
                        regions_attr: Any = getattr(field_value, "bounding_regions", None)
                        if regions_attr:
                            for region in regions_attr:
                                page_number: int = int(getattr(region, "page_number", 1))
                                polygon_raw: Any = getattr(region, "polygon", [])
                                polygon: list[float] = list(polygon_raw) if polygon_raw else []
                                if polygon:
                                    regions.append((fname_str, page_number, polygon))
    
    # Strategy 2: Fallback to figures (Layout model)
    figures_attr: Any = getattr(result, "figures", None)
    if not regions and figures_attr:
        for idx, fig in enumerate(figures_attr):
            regions_attr_fig: Any = getattr(fig, "bounding_regions", None)
            if regions_attr_fig:
                for region in regions_attr_fig:
                    page_number_fig: int = int(getattr(region, "page_number", 1))
                    polygon_raw_fig: Any = getattr(region, "polygon", [])
                    polygon_fig: list[float] = list(polygon_raw_fig) if polygon_raw_fig else []
                    if polygon_fig:
                        regions.append((f"figure_{idx+1}", page_number_fig, polygon_fig))
    
    if not regions:
        print("No signature/figure regions found")
        return {
            "file": file_path,
            "model": model_id,
            "signatures_found": 0,
            "signatures": [],
            "message": "No signature regions detected"
        }
    
    # Extract and crop signatures (IN-MEMORY)
    signatures: list[dict[str, Any]] = []
    timestamp = str(int(time.time() * 1000))
    
    # Only create output directory if saving crops is enabled
    if SAVE_CROPS:
        os.makedirs(output_dir, exist_ok=True)
    
    for i, (name, page_no, poly) in enumerate(regions, start=1):
        min_x, min_y, max_x, max_y = polygon_to_bbox(poly)
        
        # Crop signature region (in-memory)
        png_bytes = crop_image_region(
            image_bytes,
            (min_x, min_y, max_x, max_y),
            padding=PADDING_PIXELS
        )
        
        # Optionally save to disk (only if SAVE_CROPS is enabled)
        if SAVE_CROPS:
            filename = f"{Path(file_path).stem}_{name}_p{page_no}_{i}_{timestamp}.png"
            output_path = Path(output_dir) / filename
            with open(output_path, "wb") as f:
                f.write(png_bytes)
            print(f"✓ Saved: {output_path}")
        
        signatures.append({
            "index": i,
            "field_name": name,
            "page_number": page_no,
            "bbox": {"min_x": min_x, "min_y": min_y, "max_x": max_x, "max_y": max_y},
            "image_bytes": png_bytes  # In-memory PNG bytes
        })
    
    return {
        "file": file_path,
        "model": model_id,
        "signatures_found": len(signatures),
        "signatures": signatures
    }

## Example Usage - Single File

In [ ]:
# Example: Process a single image or PDF
file_path = "../testdata/valid-id/testing/valid_id1.png"  # Update with your file path

if Path(file_path).exists():
    print("="*60)
    print("AZURE DOCUMENT INTELLIGENCE - SIGNATURE EXTRACTION")
    print("="*60)
    
    result = await extract_signatures_with_doc_intelligence(file_path)
    
    print("\n" + "="*60)
    print("RESULTS")
    print("="*60)
    print(f"File: {result['file']}")
    print(f"Model: {result['model']}")
    print(f"Signatures found: {result['signatures_found']}")
    
    if result['signatures_found'] > 0:
        print("\n" + "-"*60)
        for sig in result['signatures']:
            print(f"\nSignature #{sig['index']}:")
            print(f"  Field: {sig['field_name']}")
            print(f"  Page: {sig['page_number']}")
            bbox = sig['bbox']
            print(f"  Bounding Box: ({bbox['min_x']:.1f}, {bbox['min_y']:.1f}) - ({bbox['max_x']:.1f}, {bbox['max_y']:.1f})")
            width = bbox['max_x'] - bbox['min_x']
            height = bbox['max_y'] - bbox['min_y']
            print(f"  Size: {width:.1f} x {height:.1f} pixels")
            print(f"  In-memory: image_bytes length {len(sig['image_bytes'])}")
    else:
        print(f"Message: {result.get('message', 'No signatures found')}")
else:
    print(f"❌ File not found: {file_path}")
    print("\nPlease update the file_path variable with a valid path to test.")

AZURE DOCUMENT INTELLIGENCE - SIGNATURE EXTRACTION
Processing file: ../testdata/valid-id/testing/valid_id1.png
Analyzing with model: valid_id
✓ Saved: tmp\valid_id1_customer_signature_p1_1_1762836308248.png

RESULTS
File: ../testdata/valid-id/testing/valid_id1.png
Model: valid_id
Signatures found: 1

------------------------------------------------------------

Signature #1:
  Field: customer_signature
  Page: 1
  Bounding Box: (235.0, 217.0) - (352.0, 291.0)
  Size: 117.0 x 74.0 pixels
  In-memory: image_array shape (82, 125, 3), image_bytes length 15392


## Batch Processing - Multiple Files

In [ ]:
import asyncio


async def _process_single_file(
    file_path: Path,
    model_id: str,
    output_dir: str
) -> tuple[str, dict[str, Any]]:
    """
    Process a single file asynchronously.
    
    Args:
        file_path: Path to file
        model_id: Document Intelligence model ID
        output_dir: Output directory for crops
    
    Returns:
        Tuple of (filename, result_dict)
    """
    try:
        result = await extract_signatures_with_doc_intelligence(
            str(file_path),
            model_id=model_id,
            output_dir=output_dir
        )
        print(f"✓ Success: {file_path.name} - {result['signatures_found']} signature(s) found")
        return (file_path.name, result)
    except Exception as e:
        print(f"✗ Error: {file_path.name} - {str(e)}")
        return (file_path.name, {
            "file": str(file_path),
            "error": str(e),
            "signatures_found": 0
        })


async def process_directory(
    directory_path: str = "../testdata/valid-id/testing",
    output_dir: str = "../tmp",
    model_id: str = AZURE_DI_MODEL_ID
) -> dict[str, Any]:
    """
    Process all images and PDFs in a directory IN PARALLEL using asyncio.
    
    SERVERLESS COMPATIBLE: All processing is done in-memory by default.
    File saving controlled by SAVE_CROPS environment variable.
    
    Args:
        directory_path: Path to directory containing files
        output_dir: Directory to save cropped images (only used if SAVE_CROPS=true)
        model_id: Document Intelligence model ID
    
    Returns:
        Dictionary mapping filenames to extraction results
    """
    supported_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".tiff", ".pdf"}
    
    directory = Path(directory_path)
    if not directory.exists():
        raise FileNotFoundError(f"Directory not found: {directory_path}")
    
    files = [f for f in directory.iterdir() if f.suffix.lower() in supported_extensions]
    
    print(f"\nFound {len(files)} file(s) to process in parallel\n")
    print("="*60)
    
    # Create tasks for parallel execution
    tasks = [
        _process_single_file(file, model_id, output_dir)
        for file in files
    ]
    
    # Execute all tasks in parallel with exception handling
    results_list = await asyncio.gather(*tasks, return_exceptions=True)
    
    # Convert results to dictionary with proper type guards
    results: dict[str, Any] = {}
    for result in results_list:
        if isinstance(result, Exception):
            # Handle unexpected exceptions from gather
            print(f"✗ Unexpected error: {str(result)}")
            continue
        
        if isinstance(result, tuple) and len(result) == 2:
            filename, result_dict = result
            results[filename] = result_dict
    
    print("\n" + "="*60)
    print("BATCH PROCESSING COMPLETE")
    print("="*60)
    
    total_signatures = sum(
        r.get('signatures_found', 0) for r in results.values()
    )
    successful = sum(
        1 for r in results.values() if 'error' not in r
    )
    
    print(f"\nProcessed: {len(files)} file(s)")
    print(f"Successful: {successful}")
    print(f"Failed: {len(files) - successful}")
    print(f"Total signatures extracted: {total_signatures}")
    
    return results


# Example: Process a directory (uncomment to run)
# batch_results = await process_directory("../tmp/s1-upscaled", "../tmp/s2-adi", "valid_id2")
batch_results = await process_directory("../tmp/s0-nocenter-100/UPSCALED", "../tmp/s0-nocenter-100/CROP", "valid_id2")


Found 1 file(s) to process in parallel

Processing file: ..\tmp\s0-center-100_figure_1_p1_1_1763246184812.png
Analyzing with model: valid_id2
No signature/figure regions found
✓ Success: s0-center-100_figure_1_p1_1_1763246184812.png - 0 signature(s) found

BATCH PROCESSING COMPLETE

Processed: 1 file(s)
Successful: 1
Failed: 0
Total signatures extracted: 0


## Notes

### ⚡ Serverless Compatibility
**This notebook is now SERVERLESS COMPATIBLE** following Azure Functions best practices:
- ✅ **All processing is in-memory by default** - No file system writes required
- ✅ **Optional file saving** - Controlled by `SAVE_CROPS` environment variable
- ✅ **Results always in memory** - `image_bytes` always available
- ✅ **Compatible with Azure Functions** - Can run in serverless/ephemeral environments
- ✅ **Async parallel processing** - Uses `asyncio.gather()` for concurrent file processing

**To enable saving crops to tmp/ folder:**
```python
# Set in your .env file
SAVE_CROPS=true
```

**Default behavior (serverless mode):**
```python
# All operations stay in-memory
result = await extract_signatures_with_doc_intelligence(file_path)
# Access in-memory data:
for sig in result['signatures']:
    img_bytes = sig['image_bytes']   # PNG bytes
```

**Parallel batch processing:**
```python
# Process multiple files concurrently
batch_results = await process_directory("../tmp", "../tmp/s2-adi", "valid_id2")
```

### Requirements
- Azure Document Intelligence (Form Recognizer) resource
- Environment variables in `.env` file:
  - `AZURE_DI_ENDPOINT` - Your Document Intelligence endpoint URL
  - `AZURE_DI_KEY` - Your Document Intelligence API key
  - `AZURE_DI_MODEL_ID` - Model ID (default: `prebuilt-layout`)
  - `SAVE_CROPS` - Set to `true` to save cropped images to tmp/ (optional, default: `false`)

### Supported Models
1. **`prebuilt-layout`** (default)
   - General document layout analysis
   - Detects figures, tables, text regions
   - Good for generic signature detection

2. **Custom trained models**
   - Train on your specific document types
   - Define "Signature" fields explicitly
   - More accurate for domain-specific documents
   - Set `AZURE_DI_MODEL_ID` to your custom model ID

### Tips
- **PDFs are automatically converted** to images at 200 DPI
- **Padding** is added around crops for better visibility
- **Two detection strategies**:
  1. Custom model "Signature" fields (if available)
  2. Fallback to "figures" from layout model
- For best results, use **custom models** trained on your document types
- Adjust `PDF_RENDER_DPI` and `PADDING_PIXELS` for your needs
- **Serverless mode**: Keep `SAVE_CROPS=false` for production/serverless environments
- **Local development**: Set `SAVE_CROPS=true` to save debug images to tmp/
- **Parallel processing**: Multiple files are processed concurrently using `asyncio.gather()`

### Performance
- API calls are billed based on pages analyzed
- Consider caching results for repeated analysis
- **Parallel batch processing** reduces total execution time significantly
- Custom models are faster than prebuilt models
- **In-memory processing** is faster and serverless-compatible

### Comparison with OpenAI Vision
| Feature | Azure DI | OpenAI Vision |
|---------|----------|---------------|
| **Bounding boxes** | Precise polygons | Percentage estimates |
| **Custom training** | Yes | No |
| **Cost** | Pay per page | Pay per token |
| **Speed** | Fast | Slower |
| **Accuracy** | High (trained) | Variable |
| **Setup** | Requires training | Ready to use |
| **Serverless** | ✅ Compatible | ✅ Compatible |
| **Parallel Processing** | ✅ asyncio.gather() | ✅ asyncio.gather() |

### Related Files
- Implementation in Azure Functions: `function_app.py` (see `_extract_signatures_with_doc_intelligence()`)
- OpenAI Vision approach: `signature_extraction_openai.ipynb`

### Migration from Previous Version
If upgrading from the old version:
1. **File saving is now OFF by default** - Set `SAVE_CROPS=true` if you need files saved to tmp/
2. **Results always available in memory** - No need to read saved files
3. **Async functions** - Use `await` when calling `extract_signatures_with_doc_intelligence()` and `process_directory()`
4. **Parallel processing** - `process_directory()` now processes all files concurrently
5. **Same API** - Function signatures unchanged, just different defaults and async/await